# 1_behavior_eeg_preprocess

文章提供的是预处理的数据，同时使用的也是CPz，CP1和CP2

所以可以直接使用文章提供的response-lockeddata
在perception任务中，response epoch为………………（尼玛的也不说清楚epoch的是哪个时间）

In supplementary Figs. S2–S4, we show response-locked CPPs. Those were created on the basis of the stimulus-locked CPPs, which were shifted in time such that they were all aligned to the moment of the response. 

In [2]:
import os
import numpy as np
import pandas as pd
import scipy
import matplotlib.pyplot as plt
from pathlib import Path

import logging


In [4]:
# =============================================================================
# Configuration and Constants
# =============================================================================

# Set up logging: messages will include timestamp, log level, and message content
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)

# === Paths ===
CURDIR = Path.cwd()
DS_ROOT = Path('/volumes/hyijie_psy/CPP/data_mid_van_Vugt_2019')

PATH_RESULTS =  CURDIR/'temp_data'

# Create output directories if they do not exist
os.makedirs(PATH_RESULTS, exist_ok=True)

SAMPLING_RATE = 500

SUBJECT_IDS  = [f'ACC{i:03}' for i in range(14, 37)]  # ACC014 to ACC036


In [4]:
erp_across_subjects = []
data_erp_all = []
data_beh_all = []

for subj_id in SUBJECT_IDS:
    logger.info(f'Starting prpcess sub-{subj_id}')

    # Set path for eeg and behavior data
    path_eeg = DS_ROOT / subj_id / 'CPPrespLocked.mat'
    path_behavior = DS_ROOT / subj_id / 'events.mat'
    assert path_eeg.exists(), f"EEG data file not found: {path_eeg}"
    assert path_behavior.exists(), f"events.mat not found: {path_behavior}"

    # Load eeg data from .mat files 
    eeg_resp_locked = scipy.io.loadmat(path_eeg)
    data_eeg_resp_locked = eeg_resp_locked['CPPmem']

    n_eeg = len(data_eeg_resp_locked)

    # Load behavior data
    raw_behavior = scipy.io.loadmat(path_behavior)['events'][['mode', 'type', 'rt','correct', 'item1', 'item2','cueitem']]

    data_behavior = pd.DataFrame({
        'subj_idx': subj_id,
        'mode': [extract_scalar(x) for x in raw_behavior['mode'][0]],
        'type': [extract_scalar(x) for x in raw_behavior['type'][0]],
        'rt': [extract_scalar(x) for x in raw_behavior['rt'][0]],
        'acc': [extract_scalar(x) for x in raw_behavior['correct'][0]],
        'item_1': [extract_scalar(x) for x in raw_behavior['item1'][0]],
        'item_2': [extract_scalar(x) for x in raw_behavior['item2'][0]],
        'item_cue': [extract_scalar(x) for x in raw_behavior['cueitem'][0]],
    })

    # Chose tpye and mode of bahevior data and calculate index for a certain mode
    data_behavior_resp = data_behavior[data_behavior['type'] == 'resp'].reset_index(drop=True)
    data_behavior_resp_mode = data_behavior_resp[data_behavior_resp['mode'] == 'mem']
    index_mode_unique = data_behavior_resp_mode.index
    data_behavior_resp_mode = data_behavior_resp_mode.reset_index(drop=True)

    n_behavior = len(data_behavior_resp_mode)

    # verify eeg and behavioral data have the same length
    assert n_eeg == n_behavior, f'⚠️sub{subj_id}: eeg and behavioral data are not aligned'

    df_eeg = pd.DataFrame(data_eeg_resp_locked)
    df_eeg["subject_id"] = subj_id
    data_erp_all.append(df_eeg)

    df_beh = pd.DataFrame(data_behavior_resp_mode)
    data_beh_all.append(df_beh)


data_beh_all = pd.concat(data_beh_all,  ignore_index=True)
data_erp_all = pd.concat(data_erp_all, ignore_index=True)
data_erp_all.to_csv(f'{PATH_RESULTS}/data_resp_locked_memory.csv')
data_beh_all.to_csv(f'{PATH_RESULTS}/data_beh_memory.csv')




2026-05-12 12:04:20,825 - INFO - Starting prpcess sub-ACC014
2026-05-12 12:04:20,922 - INFO - Starting prpcess sub-ACC015
2026-05-12 12:04:20,992 - INFO - Starting prpcess sub-ACC016
2026-05-12 12:04:21,061 - INFO - Starting prpcess sub-ACC017
2026-05-12 12:04:21,125 - INFO - Starting prpcess sub-ACC018
2026-05-12 12:04:21,204 - INFO - Starting prpcess sub-ACC019
2026-05-12 12:04:21,277 - INFO - Starting prpcess sub-ACC020
2026-05-12 12:04:21,353 - INFO - Starting prpcess sub-ACC021
2026-05-12 12:04:21,423 - INFO - Starting prpcess sub-ACC022
2026-05-12 12:04:21,495 - INFO - Starting prpcess sub-ACC023
2026-05-12 12:04:21,576 - INFO - Starting prpcess sub-ACC024
2026-05-12 12:04:21,638 - INFO - Starting prpcess sub-ACC025
2026-05-12 12:04:21,701 - INFO - Starting prpcess sub-ACC026
2026-05-12 12:04:21,768 - INFO - Starting prpcess sub-ACC027
2026-05-12 12:04:21,832 - INFO - Starting prpcess sub-ACC028
2026-05-12 12:04:21,897 - INFO - Starting prpcess sub-ACC029
2026-05-12 12:04:21,958 